# Customer Satisfaction Exploratory Analysis

## Análisis exploratorio de datos de satisfacción del cliente

Este notebook contiene análisis exploratorios para identificar patrones y insights en los datos de satisfacción del cliente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
%matplotlib inline

## 1. Carga de Datos

In [ ]:
# Cargar datos simulados
tickets_df = pd.read_csv('../../data/dummy/source-dummy-claude-1.csv')
surveys_df = pd.read_csv('../../data/dummy/source-dummy-claude-2.csv')
reviews_df = pd.read_csv('../../data/dummy/source-dummy-claude-3-more-negative.csv')

print(f"Tickets: {len(tickets_df):,} registros")
print(f"Encuestas: {len(surveys_df):,} registros")
print(f"Reviews: {len(reviews_df):,} registros")

## 2. Análisis de Satisfacción por Canal

In [ ]:
# Análisis de satisfacción por canal
if 'canal' in tickets_df.columns and 'satisfaccion_cliente' in tickets_df.columns:
    satisfaction_by_channel = tickets_df.groupby('canal')['satisfaccion_cliente'].agg([
        'mean', 'count', 'std'
    ]).round(2)
    
    print("Satisfacción promedio por canal:")
    print(satisfaction_by_channel)
    
    # Visualización
    fig = px.box(tickets_df, x='canal', y='satisfaccion_cliente', 
                 title='Distribución de Satisfacción por Canal')
    fig.show()

## 3. Análisis Temporal

In [ ]:
# Análisis temporal de satisfacción
if 'fecha_creacion' in tickets_df.columns:
    tickets_df['fecha_creacion'] = pd.to_datetime(tickets_df['fecha_creacion'])
    tickets_df['month'] = tickets_df['fecha_creacion'].dt.to_period('M')
    
    monthly_satisfaction = tickets_df.groupby('month')['satisfaccion_cliente'].mean()
    
    fig = px.line(x=monthly_satisfaction.index.astype(str), 
                  y=monthly_satisfaction.values,
                  title='Evolución Mensual de Satisfacción')
    fig.update_xaxes(title='Mes')
    fig.update_yaxes(title='Satisfacción Promedio')
    fig.show()

## 4. Análisis de Correlaciones

In [ ]:
# Matriz de correlación
numeric_cols = tickets_df.select_dtypes(include=[np.number]).columns
correlation_matrix = tickets_df[numeric_cols].corr()

fig = px.imshow(correlation_matrix, 
                title='Matriz de Correlación',
                color_continuous_scale='RdBu')
fig.show()

## 5. Análisis de Texto (Sentiment)

In [ ]:
# Análisis básico de sentiment en reviews
if 'texto_review' in reviews_df.columns:
    # Palabras positivas y negativas simples
    positive_words = ['excelente', 'bueno', 'satisfecho', 'recomiendo', 'positivo']
    negative_words = ['malo', 'terrible', 'insatisfecho', 'problema', 'negativo']
    
    def simple_sentiment(text):
        if pd.isna(text):
            return 'neutral'
        text_lower = text.lower()
        pos_count = sum([word in text_lower for word in positive_words])
        neg_count = sum([word in text_lower for word in negative_words])
        
        if pos_count > neg_count:
            return 'positivo'
        elif neg_count > pos_count:
            return 'negativo'
        else:
            return 'neutral'
    
    reviews_df['sentiment'] = reviews_df['texto_review'].apply(simple_sentiment)
    
    sentiment_counts = reviews_df['sentiment'].value_counts()
    
    fig = px.pie(values=sentiment_counts.values, 
                 names=sentiment_counts.index,
                 title='Distribución de Sentiment en Reviews')
    fig.show()

## 6. Insights y Conclusiones

In [ ]:
# Resumen de insights
print("📊 INSIGHTS PRINCIPALES:")
print("="*50)

if 'satisfaccion_cliente' in tickets_df.columns:
    avg_satisfaction = tickets_df['satisfaccion_cliente'].mean()
    print(f"• Satisfacción promedio general: {avg_satisfaction:.2f}/5")
    
    low_satisfaction = (tickets_df['satisfaccion_cliente'] <= 2).sum()
    total_tickets = len(tickets_df)
    print(f"• Porcentaje de clientes insatisfechos: {(low_satisfaction/total_tickets)*100:.1f}%")

if 'canal' in tickets_df.columns:
    best_channel = tickets_df.groupby('canal')['satisfaccion_cliente'].mean().idxmax()
    worst_channel = tickets_df.groupby('canal')['satisfaccion_cliente'].mean().idxmin()
    print(f"• Mejor canal: {best_channel}")
    print(f"• Canal con mayor oportunidad de mejora: {worst_channel}")

print("\n🔍 RECOMENDACIONES:")
print("• Investigar causas de insatisfacción en", worst_channel if 'worst_channel' in locals() else "canales específicos")
print("• Implementar mejores prácticas del canal", best_channel if 'best_channel' in locals() else "más exitoso")
print("• Establecer programa de monitoreo continuo")
print("• Crear alertas automáticas para satisfacción < 3")